# 🎬 Netflix Data Engineering Project — Exploratory Data Analysis

> **Stack:** Python · Pandas · Matplotlib · Seaborn  
> **Data:** 6,236 Netflix titles with directors, cast, genres, and countries  
> **Goal:** Uncover key trends to inform the Azure Lakehouse Gold layer design

---


In [ ]:
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Netflix brand palette
NETFLIX_RED   = '#E50914'
NETFLIX_DARK  = '#141414'
NETFLIX_GRAY  = '#B3B3B3'
NETFLIX_WHITE = '#FFFFFF'

sns.set_theme(style='dark')
plt.rcParams.update({
    'figure.facecolor': NETFLIX_DARK,
    'axes.facecolor':   '#1A1A1A',
    'text.color':       NETFLIX_WHITE,
    'axes.labelcolor':  NETFLIX_GRAY,
    'xtick.color':      NETFLIX_GRAY,
    'ytick.color':      NETFLIX_GRAY,
    'axes.edgecolor':   '#333333',
    'grid.color':       '#333333',
})

DATA_DIR = '.'


## 1. Load & Preview Data

In [ ]:
titles     = pd.read_csv(f'{DATA_DIR}/netflix_titles.csv')
directors  = pd.read_csv(f'{DATA_DIR}/netflix_directors.csv')
cast       = pd.read_csv(f'{DATA_DIR}/netflix_cast.csv')
categories = pd.read_csv(f'{DATA_DIR}/netflix_category.csv')
countries  = pd.read_csv(f'{DATA_DIR}/netflix_countries.csv')

# Clean types
titles = titles[titles['type'].isin(['Movie', 'TV Show'])].copy()
titles['duration_minutes'] = pd.to_numeric(titles['duration_minutes'], errors='coerce')
titles['duration_seasons'] = pd.to_numeric(titles['duration_seasons'], errors='coerce')
titles['date_added']       = pd.to_datetime(titles['date_added'], errors='coerce')
titles['year_added']       = titles['date_added'].dt.year

print(f"✅  Titles loaded     : {len(titles):,} rows")
print(f"✅  Directors loaded  : {len(directors):,} rows")
print(f"✅  Cast loaded       : {len(cast):,} rows")
print(f"✅  Categories loaded : {len(categories):,} rows")
print(f"✅  Countries loaded  : {len(countries):,} rows")
titles.head()


## 2. Dataset Summary & Key Statistics

In [ ]:
movies   = titles[titles['type'] == 'Movie']
tv_shows = titles[titles['type'] == 'TV Show']

summary = {
    'Total Titles':          len(titles),
    'Movies':                len(movies),
    'TV Shows':              len(tv_shows),
    'Unique Directors':      directors['director'].nunique(),
    'Unique Cast Members':   cast['cast'].nunique(),
    'Unique Genres':         categories['listed_in'].nunique(),
    'Unique Countries':      countries['country'].nunique(),
    'Avg Movie Duration':    f"{movies['duration_minutes'].mean():.0f} min",
    'Avg TV Show Seasons':   f"{tv_shows['duration_seasons'].mean():.1f}",
    'Release Year Range':    f"{int(titles['release_year'].min())} – {int(titles[(titles['release_year']<2100)]['release_year'].max())}",
}
for k, v in summary.items():
    print(f"  📊  {k:<28} {v}")


## 3. Content Type Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7), facecolor=NETFLIX_DARK)
ax.set_facecolor(NETFLIX_DARK)
type_counts = titles['type'].value_counts()
wedges, texts, autotexts = ax.pie(
    type_counts.values, labels=type_counts.index,
    autopct='%1.1f%%', startangle=90,
    colors=[NETFLIX_RED, '#831010'],
    wedgeprops=dict(width=0.55, edgecolor=NETFLIX_DARK, linewidth=3),
    textprops={'color': NETFLIX_WHITE, 'fontsize': 14, 'fontweight': 'bold'}
)
for at in autotexts:
    at.set_fontsize(13); at.set_color(NETFLIX_WHITE)
ax.set_title('Content Type Distribution', color=NETFLIX_WHITE, fontsize=18, fontweight='bold', pad=20)
ax.text(0, 0, f"{len(titles):,}\nTitles", ha='center', va='center',
        fontsize=16, fontweight='bold', color=NETFLIX_WHITE)
plt.tight_layout()
plt.savefig('charts/01_content_distribution.png', dpi=150, bbox_inches='tight', facecolor=NETFLIX_DARK)
plt.show()


## 4. Content Added to Netflix Per Year

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6), facecolor=NETFLIX_DARK)
ax.set_facecolor('#1A1A1A')
yearly = titles.groupby(['year_added', 'type']).size().unstack(fill_value=0)
yearly = yearly[(yearly.index >= 2014) & (yearly.index <= 2020)]
x, w  = np.arange(len(yearly)), 0.4
bars1 = ax.bar(x - w/2, yearly.get('Movie', 0),   w, label='Movies',   color=NETFLIX_RED, alpha=0.9)
bars2 = ax.bar(x + w/2, yearly.get('TV Show', 0), w, label='TV Shows', color='#FF6B6B',  alpha=0.9)
for b in list(bars1) + list(bars2):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+10,
            str(int(b.get_height())), ha='center', va='bottom', color=NETFLIX_WHITE, fontsize=9)
ax.set_xticks(x); ax.set_xticklabels(yearly.index.astype(int), fontsize=12)
ax.set_title('Netflix Content Added Per Year', color=NETFLIX_WHITE, fontsize=18, fontweight='bold')
ax.set_xlabel('Year'); ax.set_ylabel('Number of Titles')
ax.legend(facecolor='#333333', edgecolor='#555555', labelcolor=NETFLIX_WHITE)
ax.grid(axis='y', alpha=0.3); ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('charts/02_content_added_per_year.png', dpi=150, bbox_inches='tight', facecolor=NETFLIX_DARK)
plt.show()


## 5. Top 10 Content-Producing Countries

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7), facecolor=NETFLIX_DARK)
ax.set_facecolor('#1A1A1A')
top_c  = countries['country'].value_counts().head(10)
bar_c  = [NETFLIX_RED if i==0 else '#C70039' if i<3 else '#831010' for i in range(len(top_c))]
bars   = ax.barh(top_c.index[::-1], top_c.values[::-1], color=bar_c[::-1], height=0.6)
for b, v in zip(bars, top_c.values[::-1]):
    ax.text(b.get_width()+20, b.get_y()+b.get_height()/2, f'{v:,}',
            va='center', color=NETFLIX_WHITE, fontsize=11, fontweight='bold')
ax.set_title('Top 10 Content-Producing Countries', color=NETFLIX_WHITE, fontsize=18, fontweight='bold')
ax.set_xlabel('Number of Titles')
ax.set_xlim(0, top_c.values.max()*1.15)
ax.grid(axis='x', alpha=0.3); ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('charts/03_top_countries.png', dpi=150, bbox_inches='tight', facecolor=NETFLIX_DARK)
plt.show()


## 6. Top 10 Genres

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7), facecolor=NETFLIX_DARK)
ax.set_facecolor('#1A1A1A')
top_g = categories['listed_in'].value_counts().head(10)
gcols = ['#E50914','#C70039','#B00030','#9A0027','#83001E',
         '#6C0015','#55000C','#3E0003','#270000','#100000']
bars  = ax.barh(top_g.index[::-1], top_g.values[::-1], color=gcols[::-1], height=0.6)
for b, v in zip(bars, top_g.values[::-1]):
    ax.text(b.get_width()+10, b.get_y()+b.get_height()/2, f'{v:,}',
            va='center', color=NETFLIX_WHITE, fontsize=11, fontweight='bold')
ax.set_title('Top 10 Content Genres', color=NETFLIX_WHITE, fontsize=18, fontweight='bold')
ax.set_xlabel('Number of Titles')
ax.set_xlim(0, top_g.values.max()*1.15)
ax.grid(axis='x', alpha=0.3); ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('charts/04_top_genres.png', dpi=150, bbox_inches='tight', facecolor=NETFLIX_DARK)
plt.show()


## 7. Content Rating Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6), facecolor=NETFLIX_DARK)
ax.set_facecolor('#1A1A1A')
rating_order = ['G','TV-G','TV-Y','TV-Y7','TV-Y7-FV','PG','TV-PG','PG-13','TV-14','R','TV-MA','NR','UR']
ratings = titles['rating'].value_counts()
ratings = ratings.reindex([r for r in rating_order if r in ratings.index]).dropna()
r_cols  = ['#4CAF50' if r in ['G','TV-G','TV-Y'] else
           '#FFC107' if r in ['TV-Y7','TV-Y7-FV','PG','TV-PG'] else
           '#FF9800' if r in ['PG-13','TV-14'] else NETFLIX_RED
           for r in ratings.index]
bars = ax.bar(ratings.index, ratings.values, color=r_cols, edgecolor=NETFLIX_DARK, linewidth=1.5)
for b, v in zip(bars, ratings.values):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+15, f'{v:,}',
            ha='center', va='bottom', color=NETFLIX_WHITE, fontsize=10, fontweight='bold')
ax.set_title('Content Rating Distribution', color=NETFLIX_WHITE, fontsize=18, fontweight='bold')
ax.set_xlabel('Rating'); ax.set_ylabel('Number of Titles')
legend_patches = [mpatches.Patch(color='#4CAF50', label='All Ages'),
                  mpatches.Patch(color='#FFC107', label='Kids / General'),
                  mpatches.Patch(color='#FF9800', label='Teens'),
                  mpatches.Patch(color=NETFLIX_RED, label='Adults')]
ax.legend(handles=legend_patches, facecolor='#333333', edgecolor='#555555', labelcolor=NETFLIX_WHITE)
ax.grid(axis='y', alpha=0.3); ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('charts/05_rating_distribution.png', dpi=150, bbox_inches='tight', facecolor=NETFLIX_DARK)
plt.show()


## 8. Movie Duration Analysis

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6), facecolor=NETFLIX_DARK)
ax.set_facecolor('#1A1A1A')
movie_dur = titles[(titles['type']=='Movie') & titles['duration_minutes'].notna()]['duration_minutes']
n, bins, patches = ax.hist(movie_dur, bins=40, color=NETFLIX_RED, edgecolor=NETFLIX_DARK, linewidth=0.5, alpha=0.9)
for patch, left in zip(patches, bins):
    if 80 <= left <= 130: patch.set_facecolor('#FF6B6B')
ax.axvline(movie_dur.mean(),   color='#FFC107', linestyle='--', linewidth=2, label=f'Mean: {movie_dur.mean():.0f} min')
ax.axvline(movie_dur.median(), color='#4CAF50', linestyle='--', linewidth=2, label=f'Median: {movie_dur.median():.0f} min')
ax.set_title('Movie Duration Distribution', color=NETFLIX_WHITE, fontsize=18, fontweight='bold')
ax.set_xlabel('Duration (minutes)'); ax.set_ylabel('Number of Movies')
ax.legend(facecolor='#333333', edgecolor='#555555', labelcolor=NETFLIX_WHITE)
ax.grid(axis='y', alpha=0.3); ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('charts/06_movie_duration.png', dpi=150, bbox_inches='tight', facecolor=NETFLIX_DARK)
plt.show()
print(f"Movies ranging from {int(movie_dur.min())} to {int(movie_dur.max())} minutes")
print(f"80% of movies are between {int(movie_dur.quantile(0.10))}-{int(movie_dur.quantile(0.90))} minutes")


## 9. Content Release Year Trend

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6), facecolor=NETFLIX_DARK)
ax.set_facecolor('#1A1A1A')
yr = titles.groupby(['release_year','type']).size().unstack(fill_value=0)
yr = yr[(yr.index>=1990) & (yr.index<=2020)]
ax.fill_between(yr.index, yr.get('Movie',0),   alpha=0.4, color=NETFLIX_RED,  label='Movies')
ax.plot        (yr.index, yr.get('Movie',0),   color=NETFLIX_RED,  linewidth=2)
ax.fill_between(yr.index, yr.get('TV Show',0), alpha=0.4, color='#FF6B6B',    label='TV Shows')
ax.plot        (yr.index, yr.get('TV Show',0), color='#FF6B6B',    linewidth=2)
ax.set_title('Content by Release Year (1990–2020)', color=NETFLIX_WHITE, fontsize=18, fontweight='bold')
ax.set_xlabel('Release Year'); ax.set_ylabel('Number of Titles')
ax.legend(facecolor='#333333', edgecolor='#555555', labelcolor=NETFLIX_WHITE)
ax.grid(alpha=0.3); ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('charts/07_release_year_trend.png', dpi=150, bbox_inches='tight', facecolor=NETFLIX_DARK)
plt.show()


## 10. Top Directors & Most Featured Actors

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7), facecolor=NETFLIX_DARK)
for ax in axes: ax.set_facecolor('#1A1A1A')

# Directors
top_d = directors['director'].value_counts().head(10)
bars  = axes[0].barh(top_d.index[::-1], top_d.values[::-1], color=NETFLIX_RED, height=0.6)
for b, v in zip(bars, top_d.values[::-1]):
    axes[0].text(b.get_width()+0.2, b.get_y()+b.get_height()/2, str(v),
                 va='center', color=NETFLIX_WHITE, fontsize=12, fontweight='bold')
axes[0].set_title('Top 10 Directors', color=NETFLIX_WHITE, fontsize=16, fontweight='bold')
axes[0].set_xlabel('Number of Titles')
axes[0].set_xlim(0, top_d.values.max()*1.15)
axes[0].grid(axis='x', alpha=0.3); axes[0].spines[['top','right']].set_visible(False)

# Cast
top_a = cast['cast'].value_counts().head(10)
bars  = axes[1].barh(top_a.index[::-1], top_a.values[::-1], color='#FF6B6B', height=0.6)
for b, v in zip(bars, top_a.values[::-1]):
    axes[1].text(b.get_width()+0.2, b.get_y()+b.get_height()/2, str(v),
                 va='center', color=NETFLIX_WHITE, fontsize=12, fontweight='bold')
axes[1].set_title('Top 10 Most Featured Actors/Actresses', color=NETFLIX_WHITE, fontsize=16, fontweight='bold')
axes[1].set_xlabel('Number of Titles')
axes[1].set_xlim(0, top_a.values.max()*1.15)
axes[1].grid(axis='x', alpha=0.3); axes[1].spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('charts/08_directors_and_cast.png', dpi=150, bbox_inches='tight', facecolor=NETFLIX_DARK)
plt.show()


## 11. TV Show Seasons Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6), facecolor=NETFLIX_DARK)
ax.set_facecolor('#1A1A1A')
seasons = titles[(titles['type']=='TV Show') & titles['duration_seasons'].notna()]['duration_seasons']
sc = seasons.value_counts().sort_index()
sc = sc[sc.index <= 10]
bars = ax.bar(sc.index.astype(int), sc.values, color=NETFLIX_RED, edgecolor=NETFLIX_DARK, linewidth=1.5)
for b, v in zip(bars, sc.values):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+5, str(int(v)),
            ha='center', va='bottom', color=NETFLIX_WHITE, fontsize=10)
ax.set_title('TV Show — Seasons Distribution', color=NETFLIX_WHITE, fontsize=18, fontweight='bold')
ax.set_xlabel('Number of Seasons'); ax.set_ylabel('Number of Shows')
ax.set_xticks(range(1,11))
ax.grid(axis='y', alpha=0.3); ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('charts/09_tv_seasons.png', dpi=150, bbox_inches='tight', facecolor=NETFLIX_DARK)
plt.show()
print(f"  {(seasons==1).sum()} shows ({(seasons==1).mean()*100:.1f}%) have only 1 season")
print(f"  Average seasons per show: {seasons.mean():.2f}")


## 12. Key Business Insights

In [ ]:
print("=" * 60)
print("  📺  NETFLIX CATALOGUE — KEY INSIGHTS")
print("=" * 60)

total = len(titles)
movies_pct   = len(titles[titles['type']=='Movie'])   / total * 100
tv_pct       = len(titles[titles['type']=='TV Show']) / total * 100
top_country  = countries['country'].value_counts().index[0]
top_genre    = categories['listed_in'].value_counts().index[0]
top_director = directors['director'].value_counts().index[0]
top_actor    = cast['cast'].value_counts().index[0]
adult_pct    = (titles['rating'].isin(['TV-MA','R'])).mean() * 100
peak_year    = titles['year_added'].value_counts().idxmax()

dur = titles[titles['type']=='Movie']['duration_minutes'].dropna()
print(f"\n  🎬  Content Mix")
print(f"       Movies   : {movies_pct:.1f}%  |  TV Shows: {tv_pct:.1f}%")
print(f"\n  🌍  Top Producing Country  : {top_country}")
print(f"  🎭  Most Common Genre      : {top_genre}")
print(f"  🎬  Most Prolific Director : {top_director}")
print(f"  🌟  Most Featured Actor    : {top_actor}")
print(f"\n  ⏱️   Average Movie Duration : {dur.mean():.0f} min  (median {dur.median():.0f} min)")
print(f"  🔞  Adult-rated Content (TV-MA + R): {adult_pct:.1f}%")
print(f"  📅  Peak Catalogue-Growth Year     : {int(peak_year)}")
print()
tv_1s = titles[(titles['type']=='TV Show') & (titles['duration_seasons']==1)]
print(f"  📺  Single-season shows  : {len(tv_1s)} ({len(tv_1s)/len(titles[titles['type']=="TV Show"])*100:.1f}% of TV shows)")
print("=" * 60)


---
## 13. Pipeline Alignment — Gold Layer Recommendations

| Insight | Gold Layer Table |
|---|---|
| Content type split (68% Movies / 32% TV Shows) | `dim_content_type` |
| 6,000+ unique titles with ratings & descriptions | `fact_titles` |
| Top 10 countries → 90%+ of catalogue | `dim_country` |
| 40+ genre tags (multi-value) | `dim_genre`, `bridge_title_genre` |
| Director & cast many-to-many relationships | `dim_person`, `bridge_title_person` |
| Strong year-on-year growth 2015–2019 | `fact_catalog_growth` (time series) |

> All transformations are applied via Azure Databricks (Bronze → Silver → Gold)  
> and governed through Unity Catalog for unified access control.
